# Emotion Classification with RNNs, LSTMs & GRUs

Classifying text into 6 emotions (**sadness, joy, love, anger, fear, surprise**) by comparing plain recurrent models before building an **Advanced Bidirectional GRU**.

### What changed in this version
1. `Embedding(..., mask_zero=True)` so the models ignore padding (the main reason the plain RNN/GRU failed).
2. The **validation split** is used for early stopping and model selection. The **test set is used only once**, at the end.
3. `preprocess_text` is defined once here and is identical to `main.py`; it is applied to every text before tokenizing.
4. `patience` raised to 5 + `ReduceLROnPlateau`, so slow starters are not stopped before they learn.
5. A small set of hand-written sentences (not in the "i feel X" style) is added to the **training data only**.
6. Classification report, normalized confusion matrix, and sanity-check sentences with confidence.
7. Model is saved as `BiGRU_Model.keras` (matches `main.py`), plus a `versions.txt` for `requirements.txt`.

## 1. Import Libraries
Import TensorFlow/Keras, HuggingFace Datasets, and evaluation tools.

In [ ]:
import os, re, pickle, random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## 2. Load Dataset
Load the `dair-ai/emotion` dataset from Hugging Face.
https://huggingface.co/datasets/dair-ai/emotion

In [ ]:
emotion_dataset = load_dataset('dair-ai/emotion')

train_text,  train_label = emotion_dataset['train']['text'],      emotion_dataset['train']['label']
val_text,    val_label   = emotion_dataset['validation']['text'], emotion_dataset['validation']['label']
test_text,   test_label  = emotion_dataset['test']['text'],       emotion_dataset['test']['label']

In [ ]:
label_names = emotion_dataset['train'].features['label'].names

In [ ]:
label_names

In [ ]:
df_train = pd.DataFrame({'text': train_text, 'label': [label_names[i] for i in train_label]})
df_val   = pd.DataFrame({'text': val_text,   'label': [label_names[i] for i in val_label]})
df_test  = pd.DataFrame({'text': test_text,  'label': [label_names[i] for i in test_label]})
print(len(df_train), len(df_val), len(df_test))

In [ ]:
df_train.isnull().sum()

## 3. Exploratory Data Analysis (EDA)
Visualize class distribution across the 6 emotion labels.

In [ ]:
df_train['label'].value_counts().index

In [ ]:
sns.countplot(x='label', data=df_train, order=df_train['label'].value_counts().index)

In [ ]:
df_train

## 4. Data Preprocessing - Using NLP
Clean text with the shared `preprocess_text`, add a few custom training sentences, tokenize (fit on **training data only**), and pad to length 50.

In [ ]:
# IMPORTANT: keep this function IDENTICAL to preprocess_text in main.py
def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"'", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Hand-written sentences in styles the dataset lacks (not "i feel <emotion word>").
# They go into TRAINING ONLY. Add more of your own to improve real-world behaviour.
custom_sentences = {
 'sadness': ["my heart is heavy and i cant stop crying", "nobody called me on my birthday and it hurt",
             "i lost my best friend and nothing feels the same", "everything feels empty since he moved away",
             "she left and the house is so quiet now", "i failed the exam and i feel like a complete failure",
             "it was a lonely and miserable winter", "i miss my grandmother more every single day",
             "the news broke my heart", "another gloomy day and i have no energy to do anything"],
 'joy':     ["we won the match and the whole team was celebrating", "i got the job and i could not stop smiling",
             "what a wonderful day at the beach with friends", "my sister is getting married and i am thrilled",
             "this is the best news i have heard all year", "i am so proud of everything we achieved together",
             "laughing with my friends made my whole week", "i passed my driving test today and i am delighted",
             "the sunshine and music made me feel alive", "life is good and i am grateful for every moment"],
 'love':    ["i adore my wife and i cherish every moment with her", "my family means everything to me and i love them deeply",
             "i care so much about my children and their happiness", "he held my hand and i felt so tenderly cared for",
             "i am devoted to my parents and will always look after them", "seeing my dog waiting for me melts my heart",
             "i love you more than words can say", "she is the most caring and affectionate person i know",
             "we hugged for a long time and i felt completely adored", "my grandmother kisses my forehead and i feel so loved"],
 'anger':   ["i am sick of his lies and i am really angry", "how dare they cancel my flight without telling me",
             "the rude driver cut me off and i was livid", "i hate being ignored in every meeting",
             "this constant noise is driving me crazy and i am furious", "they cheated me out of my money and i am outraged",
             "stop interrupting me i am so annoyed right now", "i was fuming when i saw the damage to my car",
             "it makes my blood boil when people lie to my face", "i am bitter about how unfairly i was treated"],
 'fear':    ["i am scared to walk home alone in the dark", "my hands were shaking before the surgery",
             "i heard a strange noise downstairs and froze in terror", "i am anxious about the results of my medical test",
             "the thought of losing my job keeps me awake at night", "i was petrified when the plane started shaking",
             "i am worried something terrible will happen to my family", "the dark forest made me feel uneasy and afraid",
             "i panicked when i could not find my child in the crowd", "i dread speaking in front of a large audience"],
 'surprise':["i did not expect to see you here what a shock", "they threw me a surprise party and i was stunned",
             "i opened the box and could not believe my eyes", "wow that came completely out of nowhere",
             "i was amazed when the magician made the coin vanish", "i never imagined i would win the lottery",
             "he proposed in front of everyone and i was speechless", "what an unexpected twist i did not see that coming",
             "i was astonished by how much the town had changed", "suddenly the lights came on and everyone shouted"],
}
df_custom = pd.DataFrame([(preprocess_text(t), e) for e, ts in custom_sentences.items() for t in ts],
                         columns=['text', 'label'])
print("Custom sentences:", len(df_custom))

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len   = 50

# 1) clean every split with the same function used by the API
for df in (df_train, df_val, df_test):
    df['text'] = df['text'].apply(preprocess_text)

# 2) training data = dataset train + custom sentences repeated 8x (validation/test stay untouched)
df_train_aug = pd.concat([df_train] + [df_custom] * 8, ignore_index=True).sample(frac=1, random_state=SEED)

# 3) tokenizer is fit on TRAINING text only
tokenizer = Tokenizer(num_words=max_words, oov_token='<unk>')
tokenizer.fit_on_texts(df_train_aug['text'])

def to_padded(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=max_len, padding='post', truncating='post')

padded_train_sequence = to_padded(df_train_aug['text'])
padded_val_sequence   = to_padded(df_val['text'])
padded_test_sequence  = to_padded(df_test['text'])

train_labels = np.array([label_names.index(l) for l in df_train_aug['label']])
val_labels   = np.array([label_names.index(l) for l in df_val['label']])
test_labels  = np.array([label_names.index(l) for l in df_test['label']])

num_classes = len(label_names)
print(f"Vocabulary Size: {len(tokenizer.word_index)}")
print("Train / Val / Test:", padded_train_sequence.shape, padded_val_sequence.shape, padded_test_sequence.shape)

## 5. Shared Training Utilities
Balanced class weights, callbacks, and one helper that trains a model on the **training** set, validates on the **validation** set, and never touches the test set.

In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(train_labels),
    y            = train_labels
)

class_weight_dict = dict(enumerate(class_weights))

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def get_callbacks():
    return [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)]

def train_and_eval(model, name):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(padded_train_sequence, train_labels,
              validation_data=(padded_val_sequence, val_labels),
              epochs=20, batch_size=32, class_weight=class_weight_dict, callbacks=get_callbacks())
    loss, acc = model.evaluate(padded_val_sequence, val_labels, verbose=0)
    print(f"{name} validation loss: {loss:.4f} | validation accuracy: {acc:.4f}")
    return loss, acc

## 6. Phase 1 — Plain Foundational Models
Train plain (non-bidirectional) Simple RNN, Standard LSTM, and Standard GRU models.

### 6.1 Model 1 — Simple RNN
Plain unrolled Recurrent Neural Network.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Embedding, Dense, Dropout, LSTM, GRU

rnn_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, mask_zero=True),
    SimpleRNN(units=128, return_sequences=True),
    Dropout(0.5),
    SimpleRNN(units=64),
    Dropout(0.5),
    Dense(units=num_classes, activation='softmax')
])

In [ ]:
rnn_loss, rnn_accuracy = train_and_eval(rnn_model, 'RNN')

### 6.2 Model 2 — Standard LSTM
Plain stacked Long Short-Term Memory network.

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, mask_zero=True),
    LSTM(units=128, return_sequences=True),
    Dropout(0.5),
    LSTM(units=64),
    Dropout(0.5),
    Dense(units=num_classes, activation='softmax')
])
lstm_loss, lstm_accuracy = train_and_eval(lstm_model, 'LSTM')

### 6.3 Model 3 — Standard GRU
Plain stacked Gated Recurrent Unit network.

In [ ]:
gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, mask_zero=True),
    GRU(units=128, return_sequences=True),
    Dropout(0.5),
    GRU(units=64),
    Dropout(0.5),
    Dense(units=num_classes, activation='softmax')
])
gru_loss, gru_accuracy = train_and_eval(gru_model, 'GRU')

### 6.4 Phase 1 Ranking & Selection
Rank foundational models on the **validation** set (the test set stays untouched until the end).

In [ ]:
result_df = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU'],
    'Val Loss': [rnn_loss, lstm_loss, gru_loss],
    'Val Accuracy': [rnn_accuracy, lstm_accuracy, gru_accuracy]
}).sort_values(by='Val Accuracy', ascending=False)
result_df

## 7. Phase 2 — Advanced Bidirectional GRU
Enhance GRU with Bidirectional layers, 300D embeddings, and 0.5 dropout regularization.

### 7.1 Model 4 — Advanced Stacked Bidirectional GRU

In [ ]:
from tensorflow.keras.layers import Bidirectional, SpatialDropout1D

BiGRU = Sequential([
    Embedding(input_dim=max_words, output_dim=300, mask_zero=True),
    SpatialDropout1D(0.2),
    Bidirectional(GRU(128, return_sequences=True)),
    Dropout(0.5),
    Bidirectional(GRU(64)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')   # one probability per emotion
])
BiGRU.build(input_shape=(None, max_len))
BiGRU.summary()

#### Train & Evaluate Advanced Bidirectional GRU

In [ ]:
BiGRU_val_loss, BiGRU_val_accuracy = train_and_eval(BiGRU, 'BiGRU')

In [ ]:
# FINAL, one-time evaluation on the untouched test set
BiGRU_loss, BiGRU_accuracy = BiGRU.evaluate(padded_test_sequence, test_labels, verbose=0)
print(f"BiGRU Test Loss: {BiGRU_loss:.4f}")
print(f"BiGRU Test Accuracy: {BiGRU_accuracy:.4f}")

## 8. Overall Evaluation & Bias Analysis
Consolidate test metrics, plot raw & normalized confusion matrices, and analyze bias.

### 8.1 Classification Report & Confusion Matrices (Raw Counts & Normalized %)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

BiGRU_predictions = np.argmax(BiGRU.predict(padded_test_sequence, verbose=0), axis=1)
print(classification_report(test_labels, BiGRU_predictions, target_names=label_names, digits=3))

In [ ]:
cm = confusion_matrix(test_labels, BiGRU_predictions)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names, ax=axes[0])
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt='.0%', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[0].set_title('Raw counts'); axes[1].set_title('Normalized per true class')
for ax in axes: ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

## 9. Final Predictions
Sanity-check sentences that are **not** in the training data, including styles the dataset does not cover.

In [ ]:
sanity = [
    ("I can't believe how happy I am right now, this is amazing!", 'joy'),
    ("I feel so much love and care for my family", 'love'),
    ("My mother is the most caring person and I adore her", 'love'),
    ("How dare you lie to me again", 'anger'),
    ("I miss her so much it hurts", 'sadness'),
    ("I'm terrified of what's behind that door", 'fear'),
    ("Wow, I never saw that coming", 'surprise'),
    ("I am so grateful and delighted with this gift", 'joy'),
]
probs = BiGRU.predict(to_padded([preprocess_text(t) for t, _ in sanity]), verbose=0)

correct = 0
for (text, expected), p in zip(sanity, probs):
    top = np.argsort(p)[::-1][:2]
    ok = label_names[top[0]] == expected
    correct += ok
    print(f"{'OK ' if ok else 'BAD'} {text}\n    expected={expected:9s} predicted={label_names[top[0]]} ({p[top[0]]:.0%}), "
          f"2nd={label_names[top[1]]} ({p[top[1]]:.0%})")
print(f"\nSanity accuracy: {correct}/{len(sanity)}")

## 10. Save Model & Tokenizer
Export trained model and tokenizer artifacts for API deployment.

In [ ]:
model_dir = 'Artifacts'
os.makedirs(model_dir, exist_ok=True)

# File names match main.py
BiGRU.save(os.path.join(model_dir, 'BiGRU_Model.keras'))

with open(os.path.join(model_dir, 'tokenizer.pkl'), 'wb') as file:
    pickle.dump(tokenizer, file)

# Use these exact versions in requirements.txt so the model loads on the server
with open(os.path.join(model_dir, 'versions.txt'), 'w') as f:
    f.write(f"tensorflow=={tf.__version__}\nkeras=={keras.__version__}\nnumpy=={np.__version__}\n")

print(open(os.path.join(model_dir, 'versions.txt')).read())
print("Label order for main.py:", label_names)